# Part 1: Python & Data Foundations - Ingestion & In-Memory Validation
This notebook implements ingestion, custom validation, climate enrichment, and data cleaning for the CareLoop Clinics dataset.

In [ ]:
import csv
import json

accepted = []
rejected = []

def parse_row(row):
    parsed = row.copy()
    parsed['age'] = int(row['age'])
    parsed['fee_inr'] = int(row['fee_inr'])
    parsed['prior_no_shows'] = int(row['prior_no_shows'])
    parsed['distance_km'] = float(row['distance_km'])
    return parsed

with open('data/appointments.csv', mode='r', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        try:
            parsed = parse_row(row)
            accepted.append(parsed)
        except ValueError:
            rejected.append(row)

with open('rejected_rows.json', 'w') as f_json:
    json.dump(rejected, f_json, indent=2)
print(f'Accepted rows: {len(accepted)}, Rejected rows: {len(rejected)}')

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/appointments.csv')
df = df.drop_duplicates()
df['booking_channel'] = df['booking_channel'].astype(str).str.strip().str.title()
df['appt_date'] = pd.to_datetime(df['appt_date'], errors='coerce')
df['booked_on'] = pd.to_datetime(df['booked_on'], errors='coerce')
df = df.dropna(subset=['appt_date', 'status'])

df['age'] = pd.to_numeric(df['age'], errors='coerce')
df.loc[(df['age'] < 1) | (df['age'] > 100), 'age'] = np.nan
df['age'] = df['age'].fillna(df['age'].median()).round().astype(int)
df['distance_km'] = pd.to_numeric(df['distance_km'], errors='coerce').fillna(df['distance_km'].median())

for col in ['distance_km', 'fee_inr']:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    df[col] = np.clip(df[col], q1 - 1.5*iqr, q3 + 1.5*iqr)

df['lead_days'] = (df['appt_date'] - df['booked_on']).dt.days
df['no_show'] = np.where(df['status'] == 'No-Show', 1, 0)
df.to_csv('appointments_clean.csv', index=False)
print('Saved cleaned appointments.')

In [ ]:
import requests
clinics_df = pd.read_csv('data/clinics.csv')
temps = []
for idx, row in clinics_df.iterrows():
    try:
        res = requests.get('https://api.open-meteo.com/v1/forecast',
                           params={'latitude': row['latitude'], 'longitude': row['longitude'], 'hourly': 'temperature_2m', 'forecast_days': 1})
        temps.append(res.json()['hourly']['temperature_2m'][0])
    except:
        temps.append(None)
clinics_df['temp_c'] = temps
clinics_df.to_csv('clinics_enriched.csv', index=False)
print('Saved enriched clinics.')